# tokens

> The styling surface: colour, space, radius, type, motion, and elevation

In [ ]:
#| default_exp tokens

Tokens are the shared values that components use for styling. Changing a token changes all the components that read it. We register each family with `reg.rule` in the `tokens` layer, leaving applications free to override it with ordinary CSS.

In [ ]:
from importlib.resources import files
from fastcore.utils import *
from answerui.registry import reg, Registry

In [ ]:
from fastcore.test import *
from fasthtml.common import Div
from answerui.demo import demo


## Colour

The palette notebook produced twelve CSS custom properties: neutral and vivid ladders derived from `--hue`, with light and dark values for each. Here we give them an application default and named colour contexts.

In [ ]:
PALETTE = files('answerui').joinpath('css/palette.css').read_text()

In [ ]:
print(f'{(_ := PALETTE.splitlines())[0][:100]}…\n…\n{_[-1][:100]}…')

--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.0032…
…
--v2: light-dark(oklch(0.4111 clamp(0, 0.12402 + 0.05381*cos((1*var(--hue) - 328.74)*1deg) + 0.02534…


In [ ]:
rungs = re.findall(r'^--([\w-]+):', PALETTE, re.M)
test_eq(len(rungs), 12)


### Application palette

We use hue 260 as the default. `color-scheme: light dark` lets the browser follow the user's preference.

In [ ]:
_hue_css = ':root { --hue: 260; color-scheme: light dark; }'

### Colour contexts

Danger, warning, success, and information each get a hue and a bare attribute. These hues are independent of the application hue.

In [ ]:
CONTEXTS = dict(danger=25, warning=75, success=145, info=230)

A custom property is substituted where it is declared, so an element that sets `--hue` inherits rungs already derived from its parent's hue. A colour context therefore needs two things: `--hue` re-pointed on its selector, and the ladder declared on that same selector. A registry's `context` records both, and the palette rule is built from every context declared whenever the stylesheet is assembled, so a later module or an application can add a context without repeating the ladder:


In [ ]:
@patch(as_prop=True)
def contexts(self:Registry):
    "The colour contexts declared on this registry: selector to hue"
    return self.__dict__.setdefault('_contexts', {})

@patch
def context(
    self:Registry,
    sel, # Selector of the context
    hue, # Its hue: a number, or an expression such as `var(--hue-danger)`
):
    "Declare `sel` as a colour context whose family derives from `hue`"
    self.contexts[sel] = hue
    self.version += 1


In [ ]:
def _context_css(r): return '\n'.join(f'{sel} {{ --hue: {hue}; }}' for sel,hue in r.contexts.items())
def _palette_css(r): return f"{_context_css(r)}\n{', '.join([':root', *r.contexts])} {{\n{PALETTE}\n}}"


The four severity contexts are the first:

In [ ]:
for name in CONTEXTS: reg.context(f'[{name}]', f'var(--hue-{name})')

In [ ]:
print(_context_css(reg))

[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }


We register the application default, context hues, and palette declarations in the `tokens` layer.

In [ ]:
_hues = ' '.join(f'--hue-{name}: {hue};' for name,hue in CONTEXTS.items())
reg.rule('hue', _hue_css, layer='tokens', doc='The application hue and the supported colour schemes.')
reg.rule('contexts', f':root {{ {_hues} }}', layer='tokens', doc='The severity hues, each independently adjustable.')
reg.rule('palette', partial(_palette_css, reg), layer='tokens', doc='The neutral and vivid ladders, declared on the root and on every colour context.')


palette [tokens]: The neutral and vivid ladders, declared on the root and on every colour context.

To see the tokens, a helper binds a list of token names to one CSS property through inline styles:

In [ ]:
def swatches(prop, names, **kw):
    "Tiles binding each named token to `prop`"
    tiles = [Div(n, style=f'padding:.5rem;background:var(--n-2);font:12px monospace;{prop}:var(--{n})') for n in names]
    return Div(*tiles, style='display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)', **kw)

The ladders under the application hue, in a danger context, and as a dark island:

In [ ]:
demo(swatches('background', rungs), swatches('background', rungs, danger=True),
    Div(swatches('background', rungs), style='color-scheme:dark'))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>
<div danger style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>
<div style="color-scheme:dark">
  <div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
  </div>
</div>

```

### Tint

A control drawn as plain text, with no face of its own, shows its hover state as a tint: a faint layer of its own text colour. It is derived from `currentColor`, so the one token suits body text, the accent, or a severity context.

In [ ]:
reg.rule('tint', ':root { --tint: oklch(from currentColor l c h / .08); }', layer='tokens', doc='A faint layer of the current text colour, shown by a faceless control when hovered.')

tint [tokens]: A faint layer of the current text colour, shown by a faceless control when hovered.

In [ ]:
def tinted(label, style=''):
    "A tile tinted by its own text colour"
    return Div(label, style=f'padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);{style}')

demo(Div(tinted('Body text'), tinted('Accent', 'color:var(--v)'), Div(tinted('Danger', 'color:var(--v)'), danger=True),
    style='display:flex;gap:1rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)'))

```html
<div style="display:flex;gap:1rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);">Body text</div>
  <div style="padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);color:var(--v)">Accent</div>
  <div danger>
    <div style="padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);color:var(--v)">Danger</div>
  </div>
</div>

```

## Typography

Sizes, weights and line heights are defined here. Five sizes derive from the body size, `--text`. The base is declared on the root and the sizes on every element, so a scope that sets `--text` gets a scale of its own, and everything inside it inherits the result. Text inherits its size rather than reading the token, so such a scope also sets `font-size` from `--text`. We also declare regular and semibold weights, plus normal and tight line heights.

These are shared values, not element styles: the base layer will style native elements, while typrose will handle prose regions.

In [ ]:
_type_css = r"""
:root {
  --font: 'IBM Plex Sans', system-ui, sans-serif;
  --font-serif: 'IBM Plex Serif', Georgia, serif;
  --font-mono: 'IBM Plex Mono', ui-monospace, Menlo, Consolas, monospace;
  --text: 1rem;
  --weight: 400; --weight1: 600;
  --leading-1: 1.25; --leading: 1.5;
}
*, ::before, ::after {
  --text-1: calc(var(--text) * .875);
  --text1: calc(var(--text) * 1.25);
  --text2: calc(var(--text) * 1.5);
  --text3: calc(var(--text) * 2);
}
"""

In [ ]:
reg.rule('type', _type_css, layer='tokens',
    doc='The three font families, five sizes, regular and semibold weights, and two line heights.')

type [tokens]: The three font families, five sizes, regular and semibold weights, and two line heights.

In [ ]:
demo(swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text-1)">text-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text)">text</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text1)">text1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text2)">text2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text3)">text3</div>
</div>

```

### Fonts

We use IBM Plex: Plex Sans for text and headings, and Plex Mono for code. Plex Serif is also included, for apps that want serif headings.

The font files are shipped in the package. When an app calls `reg.serve`, they are served from the app itself. Otherwise the browser fetches them from jsDelivr.


`font_face` writes the `@font-face` rule for one file. We register all five in a single rule:


In [ ]:
FONTS = [('IBM Plex Sans', 400, 'normal'), ('IBM Plex Sans', 400, 'italic'), ('IBM Plex Sans', 600, 'normal'),
    ('IBM Plex Serif', 600, 'normal'), ('IBM Plex Mono', 400, 'normal')]

def font_face(
    family, # Family name, as CSS uses it
    weight, # Numeric weight of this file
    style,  # `normal` or `italic`
    version='5.3.0', # Version of the family's Fontsource package the public URL names
):
    "The `@font-face` rule for one packaged font file, which it registers with `reg`"
    slug = family.lower().replace(' ', '-')
    name = f'{slug}-latin-{weight}-{style}.woff2'
    url = reg.file(name, files('answerui').joinpath('fonts', name), f'https://cdn.jsdelivr.net/npm/@fontsource/{slug}@{version}/files/{name}')
    return f"@font-face {{ font-family: '{family}'; font-weight: {weight}; font-style: {style}; font-display: swap; src: url('{url}') format('woff2'); }}"

In [ ]:
reg.rule('fonts', '\n'.join(font_face(*f) for f in FONTS), layer='tokens', doc='IBM Plex Sans, Serif and Mono, from the package when served and from jsDelivr otherwise.')

fonts [tokens]: IBM Plex Sans, Serif and Mono, from the package when served and from jsDelivr otherwise.

Here are the three families. Nothing uses `--font-serif` unless an app asks for it:


In [ ]:
demo(*[Div('The quick brown fox jumps over the lazy dog, 0123456789', style=f'font-family: var(--{t})') for t in ('font', 'font-serif', 'font-mono')])


```html
<div style="font-family: var(--font)">The quick brown fox jumps over the lazy dog, 0123456789</div>
<div style="font-family: var(--font-serif)">The quick brown fox jumps over the lazy dog, 0123456789</div>
<div style="font-family: var(--font-mono)">The quick brown fox jumps over the lazy dog, 0123456789</div>

```

## Space

One scale spaces component interiors and the gaps between components. Each step derives from `--space` on every element, so changing the base on `:root` adjusts spacing across the application, and changing it on one region gives that region a scale of its own. Components use the scale for internal padding; layout parents use it for gaps, rather than components giving themselves outer margins.

The base value is the control and the steps are what components read. Setting a step directly on a region changes that element alone, since its descendants derive their own from the base they inherit.

In [ ]:
reg.rule('space', r"""
:root { --space: 1rem; }
*, ::before, ::after {
  --space-3: calc(var(--space) * .25);
  --space-2: calc(var(--space) * .5);
  --space-1: calc(var(--space) * .75);
  --space1: calc(var(--space) * 1.5);
  --space2: calc(var(--space) * 2);
  --space3: calc(var(--space) * 3);
}
""", layer='tokens', doc='A spacing scale derived from one application-wide base value.')

space [tokens]: A spacing scale derived from one application-wide base value.

In [ ]:
demo(swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-3)">space-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-1)">space-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space2)">space2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space3)">space3</div>
</div>

```

## Radius

Three sizes derive from `--radius`, on every element as with space, so changing the base adjusts the application's general roundness, or one region's. `--radius-full` stays independent: a pill is a shape, not another step on the scale.

In [ ]:
reg.rule('radius', r"""
:root { --radius: .5rem; --radius-full: 9999px; }
*, ::before, ::after {
  --radius-1: calc(var(--radius) * .5);
  --radius1: calc(var(--radius) * 2);
}
""", layer='tokens', doc='Three corner radii derived from one base value, plus an independent pill radius.')

radius [tokens]: Three corner radii derived from one base value, plus an independent pill radius.

In [ ]:
demo(swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-1)">radius-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius)">radius</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius1)">radius1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-full)">radius-full</div>
</div>

```

## Motion

`--motion` sets the shared duration, and `--ease` the easing curve. The spin that busy controls share is defined here too, so any component can turn it. Components use the duration directly or multiply it for longer effects. Temporarily increasing `--motion` lets us inspect transitions in slow motion.

A reduced-motion preference sets the duration to zero for effects that consume it. Components must remain functional without motion and must not depend on an animation finishing to perform essential work.

In [ ]:
reg.rule('motion', r"""
:root {
  --motion: 150ms;
  --ease: cubic-bezier(.2, 0, 0, 1);
}
@media (prefers-reduced-motion: reduce) {
  :root { --motion: 0ms; }
}
@keyframes aui-spin { to { rotate: 1turn; } }
""", layer='tokens', doc='A shared duration and easing curve, stilled under reduced motion, and the spin shared by busy controls.')

motion [tokens]: A shared duration and easing curve, stilled under reduced motion, and the spin shared by busy controls.

## Elevation

Depth in answerui is relief: the look of a surface lit by one light from the top left. A raised surface, such as a button, has a light rim along its top and left edges, a dark rim along its bottom and right edges, and a small shadow beneath it. A well, such as a text input, has the same lighting the other way round: dark rim at the top and left, light rim at the bottom and right, and a shadow inside its top edge. Together these give the eye a consistent reading of what stands off the page and what is cut into it.

Two values drive the whole treatment. `--face` is the colour of the surface being lit, and the rims are derived from it: a little lighter than the face for `--hi`, a little darker for `--lo`. They are derived from the face rather than taken from a rung of the ladder, because a rung that is lighter than the face in light mode is darker than it in dark mode, and a rim has to stay on the correct side of its face in both. `--relief` is the strength of the effect. At `1` the rims and shadows are fully opaque, at `.5` they are softened, and at `0` they disappear, which flattens a design without changing any face, border, or state colour. Buttons also read `--relief` to scale how far their label moves when pressed.

The derived values come as four shadow tokens that components combine. A raised surface uses `--bevel`, the rims, together with `--raised`, the shadow beneath. A well uses `--bevel-in`, the inverted rims, together with `--sunken`, the inner shadow. The shadows themselves are translucent black, so they look the same inside a colour context as outside one.

A custom property is computed where it is declared, so the rims have to be derived on the element whose face they describe, not inherited from a parent whose face may differ. The rule therefore declares them on every element, with `--face` defaulting to the raised rung, and on `::before` and `::after`, which draw the marks inside checkboxes and switches. A component whose parts are pseudo-elements of its own, such as a slider's track and thumb, declares the same derivation on them with `relief`.

Because every element derives its own rims and shadows from the two inputs, `--relief` and `--face` are the only things an application or a region needs to set. `--relief: 0` on a region flattens everything inside it; a different face on one component changes that component's rims to match. Changing the treatment of one component outright is a `box-shadow` on that component.

`--floating` is the one shadow that does not follow relief. It is the drop shadow that separates an overlay, such as a dialog or a menu, from the page beneath it, and it stays at full strength in a flat theme, because a floating panel still has to read as floating.

In [ ]:
_relief = r"""
  --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief));
  --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
  --_relief-shadow: light-dark(rgb(0 0 0 / calc(.12 * var(--relief))), rgb(0 0 0 / calc(.5 * var(--relief))));
  --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo);
  --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi);
  --raised: 0 2px 4px var(--_relief-shadow);
  --sunken: inset 0 2px 4px var(--_relief-shadow);
"""
def relief(sel):
    "The rims and shadows derived from `--face`, declared on `sel`"
    return f'{sel} {{{_relief}}}\n'

reg.rule('elevation', r"""
:root {
  --face: var(--n-2); --relief: 1;
  --_shadow-color: light-dark(rgb(0 0 0 / .12), rgb(0 0 0 / .5));
  --floating: 0 4px 12px var(--_shadow-color), 0 12px 32px var(--_shadow-color);
}
""" + relief('*, ::before, ::after'),
    layer='tokens', doc='Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.')

elevation [tokens]: Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.

In [ ]:
demo(swatches('box-shadow', ['bevel', 'raised', 'bevel-in', 'sunken', 'floating']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--bevel)">bevel</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--raised)">raised</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--bevel-in)">bevel-in</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--sunken)">sunken</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--floating)">floating</div>
</div>

```

## Customization

Application overrides use the default `app` layer, leaving the library's entries intact. Set the base values on `:root` for application-wide changes, or on a region for local ones. Colour is the exception: a local hue is a context declared with `reg.context`, since the ladder is declared per context rather than derived on every element. Here we choose a green hue, larger text, tighter spacing, and squarer corners:

In [ ]:
reg.rule('app-theme', r"""
:root { --hue: 145; --text: 1.125rem; --space: .875rem; --radius: .25rem; }
""", doc='Application-wide colour, type size, spacing, and roundness.')

app-theme [app]: Application-wide colour, type size, spacing, and roundness.

The same swatches now show the theme:

In [ ]:
demo(swatches('background', rungs), swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']),
    swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']),
    swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text-1)">text-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text)">text</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text1)">text1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text2)">text2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text3)">text3</div>
</div>
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-3)">space-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-1)">space-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space2)">space2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space3)">space3</div>
</div>
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-1)">radius-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius)">radius</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius1)">radius1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-full)">radius-full</div>
</div>

```

A region can have a scale of its own. Setting the base tokens on it rederives every step inside it. Both panels read `--text` for their font size, since text inherits its size rather than reading the token, and differ only in their inputs:


In [ ]:
def panel(title, style=''):
    "A heading, a line of text, and three spacing steps, styled from the tokens"
    return Div(Div(title, style='font-size:var(--text2);font-weight:var(--weight1)'), Div('Body text at the region\'s size.'),
        swatches('padding', ['space-2', 'space', 'space1']), style=f'font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);{style}')

demo(Div(panel('Normal'), panel('Compact', style='--space:.75rem;--text:.875rem'), style='display:grid;grid-template-columns:1fr 1fr'))

```html
<div style="display:grid;grid-template-columns:1fr 1fr">
  <div style="font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);">
    <div style="font-size:var(--text2);font-weight:var(--weight1)">Normal</div>
    <div>Body text at the region's size.</div>
    <div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
    </div>
  </div>
  <div style="font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);--space:.75rem;--text:.875rem">
    <div style="font-size:var(--text2);font-weight:var(--weight1)">Compact</div>
    <div>Body text at the region's size.</div>
    <div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
    </div>
  </div>
</div>

```

A new colour context is one declaration. Put `project=True` on a FastHTML element to give it and its contents this purple palette:


In [ ]:
reg.context('[project]', 300)


In [ ]:
demo(swatches('background', rungs, project=True))

```html
<div project style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()